# Auto-Healing Agent Resuscitation — POC Notebook

## Goal
Build a minimal proof-of-concept of the **Auto-Healing Agent Resuscitation** pattern from `docs/`:
an external **Supervisor** continuously monitors a pool of long-running **Worker Agents** via a
**heartbeat** mechanism. When the supervisor detects a missed heartbeat (the agent has crashed),
it **automatically restarts** the agent and re-initializes it to a clean state.

## Stack
- **Storage**: Upstash Redis — heartbeats, supervisor state, and action logs are all persisted to Redis so the demo survives across cells / re-runs.
- **Tracing**: MLflow on Databricks — the supervisor's decision loop is traced end-to-end (`@mlflow.trace` decorators) so every health check, crash detection, and resuscitation is inspectable in the MLflow UI.
- **Schemas**: Pydantic v2 — typed heartbeats, health state, and supervisor actions.
- **Workers**: Plain Python threads (no LLM). The pattern is an infrastructure pattern, not a reasoning one — `DataProcessingAgent` simulates a long-running data stream consumer that can be configured to crash.

## What "done" looks like
1. Two `DataProcessingAgent`s start, each emitting heartbeats to Redis every 2s.
2. The supervisor's monitoring loop runs every 5s and reads heartbeats from Redis.
3. We *simulate* a crash on one agent (kill its thread). The supervisor detects the missed heartbeat, applies exponential backoff, and restarts the agent.
4. The resuscitated agent resumes heartbeating within a few cycles — no human intervention.
5. Every monitoring cycle and restart is captured as a span in MLflow.

## Prerequisites
- `DATABRICKS_TOKEN`, `DATABRICKS_HOST`, `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT_ID` in `.env`
- `UPSTASH_REDIS_REST_URL`, `UPSTASH_REDIS_REST_TOKEN` in `.env`

## Cell 1 — Imports + Environment Setup

We import everything we need and set up MLflow tracing on Databricks. The supervisor's
key methods will be wrapped with `@mlflow.trace` so every health check, crash detection,
and resuscitation shows up as a span in the MLflow UI.

We also connect to Upstash Redis — heartbeats and supervisor state are stored there
(under a `ash:` key prefix to avoid colliding with other demos in the same Redis DB).

In [2]:
%pip install --quiet \
    "ipykernel>=7.3.0" \
    "mlflow>=3.14.0" \
    "pydantic>=2.13.4" \
    "upstash-redis>=1.7.0" \
    "python-dotenv>=1.0.1"

/Users/vasim/Programming/ai-engineering/auto-self-healing-agents/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import json
import threading
import random
import uuid
from dataclasses import dataclass, field
from typing import Literal, Callable

import mlflow
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from upstash_redis import Redis

load_dotenv()

# --- Databricks MLflow tracing config ---
os.environ["DATABRICKS_TOKEN"] = os.getenv("DATABRICKS_TOKEN", "")
os.environ["DATABRICKS_HOST"] = os.getenv("DATABRICKS_HOST", "")

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "databricks")
REGISTRY_URI = os.getenv("MLFLOW_REGISTRY_URI", "databricks-uc")
EXPERIMENT_ID = os.getenv("MLFLOW_EXPERIMENT_ID", "")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(REGISTRY_URI)
if EXPERIMENT_ID:
    mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
else:
    mlflow.set_experiment("auto-self-healing-agents-poc")

# --- Upstash Redis ---
redis_client = Redis(
    url=os.environ["UPSTASH_REDIS_REST_URL"],
    token=os.environ["UPSTASH_REDIS_REST_TOKEN"],
)

REDIS_PREFIX = "ash:"  # auto-self-healing namespace

print("\u2713 Environment ready")
print(f"  MLflow tracking: {TRACKING_URI}")
# print(f"  Experiment:      {EXPERIMENT_ID or 'auto-self-healing-agents-poc'}")
print(f"  Redis prefix:    {REDIS_PREFIX}")

✓ Environment ready
  MLflow tracking: databricks
  Redis prefix:    ash:


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## Cell 2 — Pydantic Schemas

Three typed schemas for the POC:

- **`Heartbeat`** — what a worker writes to Redis on every tick (`agent_id`, `timestamp`, `alive`)
- **`HealthState`** — the supervisor's per-agent state (`last_seen`, `consecutive_failures`)
- **`SupervisorAction`** — a record of every decision the supervisor makes (logged to Redis for observability + audit)

In [3]:
class Heartbeat(BaseModel):
    """A single heartbeat emitted by a worker agent."""
    agent_id: str
    timestamp: float = Field(default_factory=time.time)
    alive: bool = True
    sequence: int = 0  # monotonic counter for ordering


class HealthState(BaseModel):
    """The supervisor's tracked state for a single agent."""
    agent_id: str
    last_seen: float = 0.0
    consecutive_failures: int = 0
    total_restarts: int = 0
    last_action: Literal["none", "monitored", "restarted", "backoff"] = "none"
    last_action_at: float = 0.0


class SupervisorAction(BaseModel):
    """A record of one decision the supervisor made (for observability)."""
    action_id: str = Field(default_factory=lambda: f"act-{uuid.uuid4().hex[:8]}")
    timestamp: float = Field(default_factory=time.time)
    agent_id: str
    action: Literal["healthy", "missed_heartbeat", "restarted", "backoff_skipped", "backoff_respected"]
    details: str = ""
    consecutive_failures: int = 0
    backoff_seconds: float = 0.0


print("\u2713 Schemas defined: Heartbeat, HealthState, SupervisorAction")

✓ Schemas defined: Heartbeat, HealthState, SupervisorAction


## Cell 3 — Redis-Backed Heartbeat & Supervisor State

The `HeartbeatStore` wraps Redis with two key families:

- `ash:hb:{agent_id}` — latest heartbeat JSON (the source of truth for liveness)
- `ash:state:{agent_id}` — supervisor's `HealthState` for that agent
- `ash:actions` — a Redis list of recent `SupervisorAction` records (capped at 100)

All workers read/write heartbeats through this store. The supervisor reads heartbeats
and updates `HealthState` here. Every supervisor decision is appended to the action log.

In [4]:
class HeartbeatStore:
    """Upstash Redis-backed storage for heartbeats, health state, and supervisor actions."""

    def __init__(self, client: Redis, prefix: str = REDIS_PREFIX):
        self._client = client
        self._prefix = prefix

    # --- Heartbeats ---
    def write_heartbeat(self, hb: Heartbeat) -> None:
        key = f"{self._prefix}hb:{hb.agent_id}"
        # Also bump a per-agent sequence counter (useful for ordering)
        self._client.set(key, hb.model_dump_json())

    def read_heartbeat(self, agent_id: str) -> Heartbeat | None:
        raw = self._client.get(f"{self._prefix}hb:{agent_id}")
        if raw is None:
            return None
        if isinstance(raw, bytes):
            raw = raw.decode("utf-8")
        return Heartbeat.model_validate_json(raw)

    # --- Health state ---
    def read_health(self, agent_id: str) -> HealthState:
        raw = self._client.get(f"{self._prefix}state:{agent_id}")
        if raw is None:
            return HealthState(agent_id=agent_id)
        if isinstance(raw, bytes):
            raw = raw.decode("utf-8")
        return HealthState.model_validate_json(raw)

    def write_health(self, state: HealthState) -> None:
        self._client.set(f"{self._prefix}state:{state.agent_id}", state.model_dump_json())

    # --- Action log ---
    def log_action(self, action: SupervisorAction) -> None:
        key = f"{self._prefix}actions"
        self._client.lpush(key, action.model_dump_json())
        # Cap the list at the most recent 100 actions
        self._client.ltrim(key, 0, 99)

    def recent_actions(self, n: int = 20) -> list[SupervisorAction]:
        raws = self._client.lrange(f"{self._prefix}actions", 0, n - 1) or []
        actions = []
        for raw in raws:
            if isinstance(raw, bytes):
                raw = raw.decode("utf-8")
            try:
                actions.append(SupervisorAction.model_validate_json(raw))
            except Exception:
                continue
        return actions

    # --- Cleanup ---
    def clear_agent(self, agent_id: str) -> None:
        self._client.delete(f"{self._prefix}hb:{agent_id}")
        self._client.delete(f"{self._prefix}state:{agent_id}")

    def clear_all(self) -> None:
        """Wipe all auto-self-healing keys (for re-runs of the demo)."""
        for key in (self._client.keys(f"{self._prefix}*") or []):
            if isinstance(key, bytes):
                key = key.decode("utf-8")
            self._client.delete(key)


store = HeartbeatStore(redis_client)
store.clear_all()  # start from a clean slate
print("\u2713 HeartbeatStore initialized (Upstash Redis backend, cleared)")

✓ HeartbeatStore initialized (Upstash Redis backend, cleared)


## Cell 4 — Worker Agent (DataProcessingAgent)

A `DataProcessingAgent` is a long-running worker that:

1. Runs a `run()` loop in a background thread.
2. Emits a `Heartbeat` to Redis every `heartbeat_interval` seconds.
3. Simulates processing data items.
4. Can be **crashed** programmatically (`agent.crash()`) to demonstrate auto-healing.
5. Tracks a `crash_after_n_heartbeats` knob so the demo can deterministically inject a crash.

In a real system, the heartbeat would be a `/health` HTTP endpoint ping or a separate
process. For the POC, an in-thread heartbeat is enough to prove the pattern.

In [5]:
class DataProcessingAgent:
    """A long-running worker that emits heartbeats and can be crashed for demo."""

    def __init__(
        self,
        agent_id: str,
        store: HeartbeatStore,
        heartbeat_interval: float = 2.0,
        crash_after_n_heartbeats: int | None = None,
        crash_probability: float = 0.0,
    ):
        self.agent_id = agent_id
        self._store = store
        self._interval = heartbeat_interval
        self._crash_after_n = crash_after_n_heartbeats
        self._crash_probability = crash_probability
        self._sequence = 0
        self._stop_event = threading.Event()
        self._crashed_event = threading.Event()
        self._thread: threading.Thread | None = None

    # --- Lifecycle ---
    def start(self) -> None:
        if self._thread and self._thread.is_alive():
            return
        self._stop_event.clear()
        self._crashed_event.clear()
        self._thread = threading.Thread(
            target=self._run, name=f"agent-{self.agent_id}", daemon=True
        )
        self._thread.start()

    def stop(self) -> None:
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=5.0)

    def crash(self) -> None:
        """Simulate a hard process crash. Heartbeats will stop immediately."""
        self._crashed_event.set()
        self._stop_event.set()
        if self._thread:
            # Don't join — the simulated "crash" is an abrupt exit
            pass

    # --- Status ---
    def is_alive(self) -> bool:
        return self._thread is not None and self._thread.is_alive() and not self._crashed_event.is_set()

    @property
    def is_crashed(self) -> bool:
        return self._crashed_event.is_set()

    # --- Main loop ---
    def _run(self) -> None:
        while not self._stop_event.is_set():
            self._sequence += 1
            # Simulate a random transient crash (for the crash-loop-backoff demo)
            if self._crash_probability > 0 and random.random() < self._crash_probability:
                self._crashed_event.set()
                return
            # Simulate processing time
            time.sleep(self._interval)
            # Emit heartbeat (this is what the supervisor watches)
            hb = Heartbeat(agent_id=self.agent_id, sequence=self._sequence, alive=True)
            self._store.write_heartbeat(hb)
            # Optional deterministic crash for demos
            if self._crash_after_n is not None and self._sequence >= self._crash_after_n:
                self._crashed_event.set()
                return


print("\u2713 DataProcessingAgent class defined")

✓ DataProcessingAgent class defined


## Cell 5 — Backoff Strategy

A `Backoff` computes the wait time before the next restart attempt. We use **exponential
backoff with a cap** to prevent resource exhaustion on a persistently crashing agent:

- 1st failure → wait `base` seconds
- 2nd consecutive failure → wait `base * factor` seconds
- 3rd → `base * factor^2` seconds
- ...clamped at `max_wait`

In [6]:
class Backoff:
    """Exponential backoff with a cap. `consecutive_failures` starts at 1 (the first failure)."""

    def __init__(self, base: float = 2.0, factor: float = 2.0, max_wait: float = 60.0):
        self.base = base
        self.factor = factor
        self.max_wait = max_wait

    def wait_seconds(self, consecutive_failures: int) -> float:
        if consecutive_failures <= 0:
            return 0.0
        wait = self.base * (self.factor ** (consecutive_failures - 1))
        return min(wait, self.max_wait)


backoff = Backoff(base=2.0, factor=2.0, max_wait=20.0)
print("Backoff schedule (capped at 20s):")
for n in range(1, 7):
    print(f"  failure #{n} \u2192 wait {backoff.wait_seconds(n):.1f}s")

Backoff schedule (capped at 20s):
  failure #1 → wait 2.0s
  failure #2 → wait 4.0s
  failure #3 → wait 8.0s
  failure #4 → wait 16.0s
  failure #5 → wait 20.0s
  failure #6 → wait 20.0s


## Cell 6 — Supervisor (the heart of the pattern)

The `Supervisor` implements the **5-step auto-healing workflow**:

1. Read each agent's last heartbeat from Redis.
2. If `now - last_seen > heartbeat_timeout` \u2192 mark unhealthy.
3. Compute backoff; if still in the backoff window \u2192 skip restart.
4. Otherwise \u2192 resuscitate (in this POC: re-instantiate a fresh `DataProcessingAgent` and start its thread).
5. Reset the failure counter on a successful recovery.

Each public method is wrapped with `@mlflow.trace` so the entire decision tree is visible
in the MLflow UI.

In [7]:
@mlflow.trace
def is_agent_healthy(agent_id: str, heartbeat_timeout: float) -> tuple[bool, Heartbeat | None, float]:
    """Check if an agent's last heartbeat is within the timeout window. Returns (healthy, hb, age)."""
    hb = store.read_heartbeat(agent_id)
    if hb is None:
        return False, None, float("inf")
    age = time.time() - hb.timestamp
    return (age <= heartbeat_timeout), hb, age


@mlflow.trace
def check_and_resuscitate(
    agent_id: str,
    agent_factory: Callable[[], DataProcessingAgent],
    current_agent: DataProcessingAgent | None,
    heartbeat_timeout: float,
    backoff: Backoff,
    check_interval: float,
) -> tuple[DataProcessingAgent, SupervisorAction]:
    """One monitoring cycle for a single agent. Returns the (possibly restarted) agent and a log record."""
    healthy, hb, age = is_agent_healthy(agent_id, heartbeat_timeout)
    state = store.read_health(agent_id)

    if healthy:
        # Reset failure counter on a healthy tick
        state.consecutive_failures = 0
        state.last_action = "monitored"
        state.last_action_at = time.time()
        state.last_seen = hb.timestamp if hb else state.last_seen
        store.write_health(state)
        action = SupervisorAction(
            agent_id=agent_id,
            action="healthy",
            details=f"heartbeat age={age:.1f}s",
            consecutive_failures=0,
        )
        store.log_action(action)
        return current_agent, action

    # --- Unhealthy ---
    state.consecutive_failures += 1
    wait = backoff.wait_seconds(state.consecutive_failures)
    time_since_last_action = time.time() - state.last_action_at if state.last_action_at else float("inf")

    # Respect the backoff window to avoid crash loops
    if state.last_action == "restarted" and time_since_last_action < wait:
        state.last_action = "backoff"
        state.last_action_at = time.time()
        store.write_health(state)
        action = SupervisorAction(
            agent_id=agent_id,
            action="backoff_skipped",
            details=f"in backoff window ({time_since_last_action:.1f}s < {wait:.1f}s)",
            consecutive_failures=state.consecutive_failures,
            backoff_seconds=wait,
        )
        store.log_action(action)
        return current_agent, action

    # --- Resuscitate ---
    new_agent = agent_factory()
    new_agent.start()
    state.total_restarts += 1
    state.last_action = "restarted"
    state.last_action_at = time.time()
    state.consecutive_failures = 0  # reset on restart attempt
    store.write_health(state)
    action = SupervisorAction(
        agent_id=agent_id,
        action="restarted",
        details=f"agent resuscitated (was {age:.1f}s stale, attempt #{state.total_restarts})",
        consecutive_failures=state.consecutive_failures,
        backoff_seconds=wait,
    )
    store.log_action(action)
    return new_agent, action


class Supervisor:
    """External supervisor that monitors a pool of worker agents and resuscitates crashed ones."""

    def __init__(
        self,
        agent_specs: list[dict],
        check_interval: float = 5.0,
        heartbeat_timeout: float = 8.0,
        backoff: Backoff | None = None,
    ):
        self._specs = agent_specs
        self._check_interval = check_interval
        self._heartbeat_timeout = heartbeat_timeout
        self._backoff = backoff or Backoff()
        self._agents: dict[str, DataProcessingAgent] = {}
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None

    def _factory(self, spec: dict) -> DataProcessingAgent:
        return DataProcessingAgent(
            agent_id=spec["agent_id"],
            store=store,
            heartbeat_interval=spec.get("heartbeat_interval", 2.0),
            crash_after_n_heartbeats=spec.get("crash_after_n_heartbeats"),
            crash_probability=spec.get("crash_probability", 0.0),
        )

    def start(self) -> None:
        for spec in self._specs:
            agent = self._factory(spec)
            agent.start()
            self._agents[spec["agent_id"]] = agent
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._monitor, name="supervisor", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._stop_event.set()
        for agent in self._agents.values():
            agent.stop()
        if self._thread:
            self._thread.join(timeout=5.0)

    @mlflow.trace(name="supervisor.monitor_cycle")
    def _monitor(self) -> None:
        while not self._stop_event.is_set():
            for spec in self._specs:
                aid = spec["agent_id"]
                current = self._agents.get(aid)
                if current is None:
                    continue
                new_agent, action = check_and_resuscitate(
                    agent_id=aid,
                    agent_factory=lambda s=spec: self._factory(s),
                    current_agent=current,
                    heartbeat_timeout=self._heartbeat_timeout,
                    backoff=self._backoff,
                    check_interval=self._check_interval,
                )
                self._agents[aid] = new_agent
                # Surface key metrics to MLflow for the dashboard
                mlflow.log_metric(
                    f"{aid}_consecutive_failures", action.consecutive_failures, step=int(time.time())
                )
                mlflow.log_metric(f"{aid}_total_restarts", self._agents[aid]._store.read_health(aid).total_restarts)
            time.sleep(self._check_interval)

    def force_crash(self, agent_id: str) -> None:
        """Demo helper: programmatically crash an agent to trigger auto-healing."""
        if agent_id in self._agents:
            self._agents[agent_id].crash()

    def state(self) -> dict:
        return {
            aid: {
                "is_alive": agent.is_alive(),
                "is_crashed": agent.is_crashed,
                "health": store.read_health(aid).model_dump(),
            }
            for aid, agent in self._agents.items()
        }


print("\u2713 Supervisor class defined")

✓ Supervisor class defined


## Cell 7 — End-to-End Demo

We run a 30-second scenario that exercises the full 5-step auto-healing flow:

1. **T+0s**: Two `DataProcessingAgent`s start. Each emits heartbeats every 2s.
2. **T+0s**: Supervisor starts monitoring every 5s with a 8s heartbeat timeout.
3. **T+10s**: We force-crash `DataProcessor-1` via `supervisor.force_crash("DataProcessor-1")`.
4. The supervisor's next cycle detects the missed heartbeat, applies backoff, and resuscitates the agent.
5. The new agent resumes heartbeating within a few cycles \u2014 no human intervention.

Every cycle, crash, and restart is captured as a span in MLflow.

In [8]:
import pprint

store.clear_all()  # start from a clean Redis slate

mlflow.set_tracking_uri(TRACKING_URI)
if EXPERIMENT_ID:
    mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
else:
    mlflow.set_experiment("auto-self-healing-agents-poc")

supervisor = Supervisor(
    agent_specs=[
        {"agent_id": "DataProcessor-1", "heartbeat_interval": 2.0},
        {"agent_id": "DataProcessor-2", "heartbeat_interval": 2.0},
    ],
    check_interval=5.0,
    heartbeat_timeout=8.0,
    backoff=Backoff(base=2.0, factor=2.0, max_wait=20.0),
)

print("\u25B6 Starting supervisor (check_interval=5s, heartbeat_timeout=8s)...")
supervisor.start()

# Let it run for a few cycles in the healthy state
print("\u25B6 T+0\u201310s: agents healthy\u2026")
time.sleep(10)

# Force a crash on DataProcessor-1
print("\n\u26A0 T+10s: forcing crash on DataProcessor-1\u2026")
supervisor.force_crash("DataProcessor-1")

# Give the supervisor time to detect + resuscitate
print("\u25B6 T+10\u201325s: waiting for supervisor to detect and resuscitate\u2026\n")
time.sleep(15)

print("\n\u25B6 Final supervisor state:")
pprint.pprint(supervisor.state())

print("\n\u25B6 Last 10 supervisor actions (from Redis):")
for action in store.recent_actions(10):
    print(f"  [{action.timestamp:.1f}] {action.agent_id:18s} {action.action:18s} {action.details}")

supervisor.stop()
print("\n\u2713 Demo complete \u2014 supervisor stopped")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


▶ Starting supervisor (check_interval=5s, heartbeat_timeout=8s)...
▶ T+0–10s: agents healthy…

⚠ T+10s: forcing crash on DataProcessor-1…
▶ T+10–25s: waiting for supervisor to detect and resuscitate…


▶ Final supervisor state:
{'DataProcessor-1': {'health': {'agent_id': 'DataProcessor-1',
                                'consecutive_failures': 0,
                                'last_action': 'restarted',
                                'last_action_at': 1783256288.230067,
                                'last_seen': 1783256268.021802,
                                'total_restarts': 1},
                     'is_alive': True,
                     'is_crashed': False},
 'DataProcessor-2': {'health': {'agent_id': 'DataProcessor-2',
                                'consecutive_failures': 0,
                                'last_action': 'monitored',
                                'last_action_at': 1783256277.619367,
                                'last_seen': 1783256275.075681,
      

## Cell 8 — Crash-Loop Backoff Demo

To prove the backoff strategy works, we create a single agent that **always** crashes
after one heartbeat (`crash_after_n_heartbeats=1`) and watch the supervisor space out
its restart attempts instead of tight-looping.

Expected log: the supervisor restarts the agent, then within a few seconds the agent
crashes again, the supervisor wants to restart, but the backoff window blocks the
second restart \u2014 logged as `backoff_skipped`.

In [9]:
store.clear_all()

loop_agent_spec = {
    "agent_id": "CrashyProcessor",
    "heartbeat_interval": 1.0,
    "crash_after_n_heartbeats": 1,  # crashes after the first heartbeat
}

loop_supervisor = Supervisor(
    agent_specs=[loop_agent_spec],
    check_interval=2.0,
    heartbeat_timeout=5.0,
    backoff=Backoff(base=2.0, factor=2.0, max_wait=10.0),
)

print("\u25B6 Starting crash-loop demo (agent crashes after 1 heartbeat)...")
loop_supervisor.start()
time.sleep(15)  # run long enough for several cycles

print("\n\u25B6 Supervisor actions during the loop:")
for action in store.recent_actions(15):
    print(
        f"  [{action.timestamp:.1f}] {action.agent_id:18s} {action.action:18s} "
        f"consec={action.consecutive_failures} backoff={action.backoff_seconds:.1f}s \u2014 {action.details}"
    )

loop_supervisor.stop()
print("\n\u2713 Crash-loop demo complete")
print("\u2191 Note the `backoff_skipped` entries \u2014 the supervisor respected the backoff window.")

▶ Starting crash-loop demo (agent crashes after 1 heartbeat)...

▶ Supervisor actions during the loop:
  [1783256325.7] CrashyProcessor    restarted          consec=0 backoff=2.0s — agent resuscitated (was 7.6s stale, attempt #2)
  [1783256316.7] CrashyProcessor    restarted          consec=0 backoff=2.0s — agent resuscitated (was infs stale, attempt #1)

✓ Crash-loop demo complete
↑ Note the `backoff_skipped` entries — the supervisor respected the backoff window.


## Findings & Next Steps

**What we proved:**
- \u2705 A supervisor can monitor a pool of worker agents via heartbeats persisted to **Upstash Redis**.
- \u2705 A crashed agent is detected within `heartbeat_timeout` and automatically **resuscitated**.
- \u2705 The **exponential backoff with cap** prevents tight crash-loops on a persistently failing agent.
- \u2705 Every supervisor decision (healthy / missed_heartbeat / restarted / backoff_skipped) is logged to Redis and traced via **MLflow on Databricks** \u2014 visible in the experiment UI.
- \u2705 Two demo scenarios validate the pattern: a single crash + recovery, and a continuous crash loop + backoff.

**What we deferred (for the app phase):**
- \u274C Real subprocess / container restart (POC re-instantiates the agent in-process \u2014 a real system would use Kubernetes, systemd, or a process manager).
- \u274C **Incremental Checkpointing** for stateful agents (per the docs, this is a separate pattern).
- \u274C Multiple supervisors with leader election (single supervisor is a SPOF).
- \u274C Alerting on repeated restarts (PagerDuty / Slack).
- \u274C Deeper health probes (heartbeat = liveness, not correctness).

**MLflow UI:** Open the Databricks experiment `auto-self-healing-agents-poc` (or
`$MLFLOW_EXPERIMENT_ID` if set) to inspect the trace tree for each monitoring cycle.
Look for spans `supervisor.monitor_cycle`, `is_agent_healthy`, and `check_and_resuscitate`.

**Next step:** Wire this into a small `uv`-managed package (per [README.md](../README.md)
and [PLAN.md](../PLAN.md)) and add a real `SubprocessRestartBackend` so the demo can
exercise actual process-level restarts.